In [203]:
import pandas as pd
import seaborn as sns
import json
import numpy as np
import matplotlib.pyplot as plt

In [204]:
# Set Seaborn style
sns.set_style("whitegrid")

In [205]:
with open("../data/raw/bootstrap.json") as f:
        data = json.load(f)

In [206]:
players = pd.DataFrame(data["elements"])

In [207]:
teams = pd.DataFrame(data['teams'])[['id', 'code', 'name', 'short_name']]

In [208]:
#Merge some parts of team with players based on id
players = players.merge(teams, left_on='team', right_on='id', how='left')

In [209]:
players.columns.to_list()

In [210]:
print(f" Number of total players: {players.shape[0]}")

In [211]:
df = players.copy()

### Seeing the nations most represented (Having played atleast 1 minute)

In [212]:
df = df[df['minutes'] > 0]

In [213]:
print(f"Number of unique nations represented: {df['region'].nunique()}")

In [214]:
df['region'].value_counts()

In [215]:
df.groupby('region', as_index=False).last()[['region', 'web_name','name']]

In [216]:
regions_name = {
    2: 'Albania',
    3: 'Algeria',
    10: 'Argentina',
    14: 'Austria',
    21: 'Belgium',
    30: 'Brazil',
    34: 'Bulgaria',
    35: 'Burkina Faso',
    38: 'Cameroon',
    48: 'Colombia',
    50: 'DR Congo',
    54: 'Cote d\'Ivoire',
    57: 'Czech Republic',
    58: 'Denmark',
    62: 'Ecuador',
    63: 'Egypt',
    73: 'France',
    78: 'Gambia',
    79: 'Georgia',
    80: 'Germany',
    81: 'Ghana',
    83: 'Greece',
    90: 'Guinea-Bissau',
    92: 'France', 
    97: 'Spain',
    98: 'Hungary',
    104: 'Republic of Ireland',
    106: 'Italy',
    107: 'Jamaica',
    108: 'Japan',
    114: 'South Korea',
    139: 'Mexico',
    145: 'Morocco',
    146: 'Mozambique',
    152: 'Netherlands',
    154: 'New Zealand',
    157: 'Nigeria',
    161: 'Norway',
    168: 'Paraguay',
    169: 'Peru',
    172: 'Poland',
    173: 'Portugal',
    189: 'Senegal',
    190: 'Serbia',
    194: 'Slovakia',
    195: 'Slovenia',
    198: 'South Africa',
    200: 'Spain',
    206: 'Sweden',
    207: 'Switzerland',
    218: 'Tunisia',
    219: 'Turkey',
    225: 'Ukraine',
    229: 'USA',
    230: 'Uruguay',
    231: 'Uzbekistan',
    239: 'Zimbabwe',
    241: 'England',
    242: 'Northern Ireland',
    243: 'Scotland',
    244: 'Wales'
}

In [217]:
df['country'] = df['region'].map(regions_name)

In [218]:
country_counts = df['country'].value_counts()
country_counts

In [219]:
# Get countries with exactly 1 player
single_player_countries = country_counts[(country_counts == 1) | (country_counts == 2)].index
df_single_players = df[df['country'].isin(single_player_countries)][['web_name', 'name', 'country']]
df_single_players.sort_values(by='country')

In [220]:
# 5 Oldest players
df[['web_name','name','birth_date']].sort_values(by='birth_date').head(5)

In [221]:
# 5 Youngest players
df[['web_name','name','birth_date']].sort_values(by='birth_date',ascending=False).head(5)

In [222]:
# Players sharing the same Date of Birth
players_dob_counts = df['birth_date'].value_counts()
max_count = players_dob_counts.max()

In [223]:
# 2 dates on which 3 players were born on the same date
most_common_birthdates = players_dob_counts[players_dob_counts == max_count].index
df_same_bday = df[df['birth_date'].isin(most_common_birthdates)][['web_name', 'name', 'birth_date']].sort_values(by='birth_date')
df_same_bday

In [224]:
# Year histogram

df['birth_date'] = pd.to_datetime(df['birth_date'])

# Extract year, month, day
df['birth_year'] = df['birth_date'].dt.year
df['birth_month'] = df['birth_date'].dt.month
df['birth_day'] = df['birth_date'].dt.day


plt.figure(figsize=(14,5))
sns.histplot(df['birth_year'], bins=np.arange(df['birth_year'].min(), df['birth_year'].max()+1),
             kde=False, color='skyblue')
plt.title("Player Birth Year Distribution")
plt.xlabel("Year")
plt.ylabel("Number of Players")
plt.xticks(np.arange(df['birth_year'].min(), df['birth_year'].max()+1), rotation=45)
plt.show()

# Month histogram
plt.figure(figsize=(12,4))
sns.histplot(df['birth_month'], bins=np.arange(1,14), kde=False, color='lightgreen')
plt.title("Player Birth Month Distribution")
plt.xlabel("Month")
plt.ylabel("Number of Players")
plt.xticks(np.arange(1,13))
plt.show()

# Day histogram
plt.figure(figsize=(12,4))
sns.histplot(df['birth_day'], bins=np.arange(1,32), kde=False, color='salmon')
plt.title("Player Birth Day Distribution")
plt.xlabel("Day of Month")
plt.ylabel("Number of Players")
plt.xticks(np.arange(1,32))
plt.show()

In [225]:
# Describe year, month, day and combine nicely
summary = pd.concat([
    df['birth_year'].describe(),
    df['birth_month'].describe(),
    df['birth_day'].describe()
], axis=1)

summary.columns = ['Year', 'Month', 'Day']
summary

In [231]:
sns.boxplot(df['birth_year'])

In [232]:
sns.boxplot(df['birth_month'])